In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

df = pd.read_csv("../data_processed/02_catalogo_limpio_base.csv")

df.shape

(6267, 35)

In [13]:
# Densidad reportada directamente por el NASA Exoplanet Archive
df["densidad_reportada"] = df["pl_dens"]

# Densidad calculada a partir de masa y radio en unidades terrestres
# rho = rho_tierra * M / R^3
# rho_tierra = 5.51 g/cm^3

df["densidad_calculada"] = 5.51 * df["pl_bmasse"] / (df["pl_rade"] ** 3)

df[["pl_name", "pl_bmasse", "pl_rade", "densidad_reportada", "densidad_calculada"]].head()

,pl_name,pl_bmasse,pl_rade,densidad_reportada,densidad_calculada
0,11 Com b,4914.898486,12.2,14.90,14.913743
1,11 UMi b,4684.814200,12.3,13.80,13.871666
2,14 And b,1131.151301,13.1,2.76,2.772416
3,14 Her b,2828.672822,12.5,7.96,7.980025
4,16 Cyg B b,565.737400,13.5,1.26,1.266967


In [14]:
def clasificar_radio(r):
    if pd.isna(r):
        return "sin_dato"
    elif r < 1.25:
        return "terrestre"
    elif r < 2.0:
        return "supertierra"
    elif r < 6.0:
        return "neptuniano"
    elif r < 15.0:
        return "gigante_gaseoso"
    else:
        return "mayor_que_jupiter"

df["clase_radio"] = df["pl_rade"].apply(clasificar_radio)

df["clase_radio"].value_counts()

clase_radio
neptuniano           2374
gigante_gaseoso      1853
supertierra          1188
terrestre             571
mayor_que_jupiter     237
sin_dato               44
Name: count, dtype: int64

In [15]:
def clasificar_masa(m):
    if pd.isna(m):
        return "sin_dato"
    elif m < 2:
        return "masa_terrestre"
    elif m < 10:
        return "supertierra_masa"
    elif m < 50:
        return "subneptuno_neptuno_masa"
    elif m < 4000:
        return "gigante_masa"
    else:
        return "muy_masivo"

df["clase_masa"] = df["pl_bmasse"].apply(clasificar_masa)

df["clase_masa"].value_counts()

clase_masa
supertierra_masa           2741
gigante_masa               1879
subneptuno_neptuno_masa     950
masa_terrestre              507
muy_masivo                  164
sin_dato                     26
Name: count, dtype: int64

In [16]:
def clasificar_masa(m):
    if pd.isna(m):
        return "sin_dato"
    elif m < 2:
        return "masa_terrestre"
    elif m < 10:
        return "supertierra_masa"
    elif m < 50:
        return "subneptuno_neptuno_masa"
    elif m < 4000:
        return "gigante_masa"
    else:
        return "muy_masivo"

df["clase_masa"] = df["pl_bmasse"].apply(clasificar_masa)

df["clase_masa"].value_counts()

clase_masa
supertierra_masa           2741
gigante_masa               1879
subneptuno_neptuno_masa     950
masa_terrestre              507
muy_masivo                  164
sin_dato                     26
Name: count, dtype: int64

In [17]:
def clasificar_periodo(p):
    if pd.isna(p):
        return "sin_dato"
    elif p < 1:
        return "USP"
    elif p < 10:
        return "periodo_corto"
    elif p < 100:
        return "periodo_intermedio"
    else:
        return "periodo_largo"

df["clase_periodo"] = df["pl_orbper"].apply(clasificar_periodo)

df["clase_periodo"].value_counts()

clase_periodo
periodo_corto         2695
periodo_intermedio    2100
periodo_largo          973
sin_dato               340
USP                    159
Name: count, dtype: int64

In [18]:
df["es_pequeno"] = df["pl_rade"] < 2.0

df["es_gigante"] = df["pl_rade"] >= 6.0

df["es_hot_jupiter"] = (
    (df["pl_rade"] >= 6.0) &
    (df["pl_orbper"] < 10)
)

df["es_super_puff"] = (
    (df["pl_rade"] >= 4.0) &
    (df["densidad_calculada"] < 0.3)
)

In [19]:
conteo_radio = df["clase_radio"].value_counts().reset_index()
conteo_radio.columns = ["clase_radio", "numero_de_planetas"]

conteo_masa = df["clase_masa"].value_counts().reset_index()
conteo_masa.columns = ["clase_masa", "numero_de_planetas"]

conteo_periodo = df["clase_periodo"].value_counts().reset_index()
conteo_periodo.columns = ["clase_periodo", "numero_de_planetas"]

conteo_indicadores = pd.DataFrame({
    "indicador": ["es_pequeno", "es_gigante", "es_hot_jupiter", "es_super_puff"],
    "numero_de_planetas": [
        df["es_pequeno"].sum(),
        df["es_gigante"].sum(),
        df["es_hot_jupiter"].sum(),
        df["es_super_puff"].sum()
    ]
})

conteo_radio, conteo_masa, conteo_periodo, conteo_indicadores

(         clase_radio  numero_de_planetas
 0         neptuniano                2374
 1    gigante_gaseoso                1853
 2        supertierra                1188
 3          terrestre                 571
 4  mayor_que_jupiter                 237
 5           sin_dato                  44,
                 clase_masa  numero_de_planetas
 0         supertierra_masa                2741
 1             gigante_masa                1879
 2  subneptuno_neptuno_masa                 950
 3           masa_terrestre                 507
 4               muy_masivo                 164
 5                 sin_dato                  26,
         clase_periodo  numero_de_planetas
 0       periodo_corto                2695
 1  periodo_intermedio                2100
 2       periodo_largo                 973
 3            sin_dato                 340
 4                 USP                 159,
         indicador  numero_de_planetas
 0      es_pequeno                1759
 1      es_gigante             

In [20]:
conteo_radio.to_csv("../tables/conteo_clase_radio.csv", index=False)
conteo_masa.to_csv("../tables/conteo_clase_masa.csv", index=False)
conteo_periodo.to_csv("../tables/conteo_clase_periodo.csv", index=False)
conteo_indicadores.to_csv("../tables/conteo_indicadores_planetarios.csv", index=False)

In [21]:
df.to_csv("../data_processed/03_catalogo_con_parametros_derivados.csv", index=False)

df.shape

(6267, 44)

In [22]:
df["clase_radio"].value_counts()
df["clase_masa"].value_counts()
df["clase_periodo"].value_counts()
conteo_indicadores

,indicador,numero_de_planetas
0,es_pequeno,1759
1,es_gigante,2090
2,es_hot_jupiter,792
3,es_super_puff,230
